[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_10/4_ai.ipynb)

# Day 10: AI and ML (easy)

**How to work:** Answer concept questions out loud in 60 to 90 seconds before opening the solution: definition, why it matters, one example, one trade-off.

**Today (26 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Groups without labels | easy | 4 |
| Q2 | Can k-means find the species? | easy | 7 |
| Q3 | Thirty columns, how many directions? | easy | 6 |
| Q4 | One step of the algorithm | easy | 6 |
| Q5 | Prepare customers for clustering (review) | easy | 3 |

**Handbook:** [AI and ML](https://mahsa-agz.github.io/ds-handbook/#ai), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `penguins`: 344 Palmer penguins. `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex` (MALE/FEMALE, has NaN).
- `load_breast_cancer()` from scikit-learn (built in): 569 tumours, 30 numeric features on very different scales.

The setup cell loads every table into a pandas DataFrame (from your Drive folder, or downloads it). Everything runs on the free Colab CPU in seconds. Numbers in the solutions come from scikit-learn 1.8; other versions can differ in the last digit.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import io, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)

SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def load_telco():
    """IBM Telco churn, 7,043 customers. Target: Churn (Yes/No)."""
    return pd.read_csv(data_path("telco_churn.csv", TELCO_URL))

def load_seaborn(name):
    """titanic, penguins, diamonds, mpg, tips ... from the seaborn-data repo."""
    return pd.read_csv(data_path(name + ".csv", SEABORN + name + ".csv"))

def load_bank():
    """UCI Bank Marketing (bank-additional-full), 41,188 calls. Target: y (yes/no). The source is a zip in a zip."""
    path = data_path("bank_marketing.csv")
    if not os.path.exists(path):
        print("downloading bank_marketing (zip)")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(BANK_URL).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with open(path, "wb") as f:
            f.write(inner.read("bank-additional/bank-additional-full.csv"))
    return pd.read_csv(path, sep=";")

def check(fn, cases, tol=1e-6):
    """Runs fn on each (args, expected) case; numbers and arrays are compared with a tolerance."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*args)
            good = got is not None and np.shape(got) == np.shape(want) and bool(
                np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), atol=tol))
        except Exception as e:
            got, good = "error: " + repr(e)[:120], False
        ok += good
        shown = " ".join(repr(got).split())
        shown = shown if len(shown) < 100 else shown[:97] + "..."
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}(...) -> {shown}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

penguins = load_seaborn("penguins")
print("penguins", penguins.shape)

---
### Q1. Groups without labels

*easy, about 4 min*

Explain k-means in about 90 seconds: the objective, the algorithm, how you pick k, and three situations where it fails. Finish with one sentence on how PCA differs from clustering.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: 'segment users, no labels'. Pattern: k-means (minimise within-cluster squared distance) and its assumptions.

</details>

<details><summary><b>Hint 2</b></summary>

1. Objective: inertia `sum ||x - mu_c(x)||^2`. 2. Lloyd's algorithm: assign to nearest centre, move centres to the mean, repeat. 3. k-means++ init, several restarts. 4. Choose k: elbow, silhouette, business use. 5. Fails: unscaled features, non-spherical or unequal clusters, outliers, categorical data.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 90 s):**

"k-means splits the data into k groups to minimise the within-cluster sum of squared distances, the inertia `sum_i ||x_i - mu_c(i)||^2`. Lloyd's algorithm alternates two steps: assign each point to its nearest centre, then move each centre to the mean of its points. Each step lowers the inertia, so it converges, but only to a local optimum; so we use k-means++ initialisation (spread-out starting centres) and several restarts. Each iteration costs `O(n k d)`.

To choose k: the elbow of inertia versus k, the silhouette score, the stability of clusters across samples, and above all whether the segments are usable by the business.

It fails when features are on different scales (the largest unit dominates the distance), when clusters are elongated, nested or of very different sizes or densities (it assumes round, similar-size blobs), with outliers (means get pulled), and with categorical data (use k-modes or Gower distance). Alternatives: Gaussian mixtures, DBSCAN, hierarchical clustering.

PCA is different: it does not group rows, it finds new axes (directions of maximum variance) to compress the columns."

**Why:** The objective, the two alternating steps, local optima with k-means++, how to choose k, and the scaling/shape assumptions cover what examiners check.

**Common mistakes:** Saying k-means finds the global optimum. Forgetting scaling. Choosing k only by the elbow when the curve has no clear elbow.

**Learn more:** [ai-unsupervised](https://mahsa-agz.github.io/ds-handbook/#ai-unsupervised)

</details>

---
### Q2. Can k-means find the species?

*easy, about 7 min*

Use `penguins` with the four body measurements (drop rows where any is missing). Pretend you do not know `species`.

1. Run `KMeans(n_clusters=3, n_init=10, random_state=0)` on the **raw** measurements, and again on **standardised** measurements.
2. Compare each clustering with the true species using the adjusted Rand index (ARI: 1 = perfect match, about 0 = random).
3. Print a crosstab of species versus the scaled clusters.

Why does scaling matter so much here? Which species get mixed up?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: clustering with features in different units (mm and grams). Pattern: k-means needs scaled features; evaluate against known labels with ARI.

</details>

<details><summary><b>Hint 2</b></summary>

1. `KMeans(...).fit_predict(X)`. 2. `StandardScaler().fit_transform(X)`. 3. `adjusted_rand_score(true, pred)` ignores how clusters are numbered. 4. `pd.crosstab`.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
d = penguins.dropna(subset=cols)
print("rows:", len(d))                                           # out: rows: 342

raw_labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(d[cols])
Z = StandardScaler().fit_transform(d[cols])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
print("ARI unscaled:", round(adjusted_rand_score(d["species"], raw_labels), 3))   # out: ARI unscaled: 0.331
print("ARI scaled:  ", round(adjusted_rand_score(d["species"], km.labels_), 3))    # out: ARI scaled:   0.793
print(pd.crosstab(d["species"], km.labels_, colnames=["cluster"]).to_string())
# (cluster numbers are arbitrary and can differ between versions)
# out: cluster      0    1   2
# out: species
# out: Adelie       0  127  24
# out: Chinstrap    0    5  63
# out: Gentoo     123    0   0
```

**Why:** On raw data, `body_mass_g` (in grams, standard deviation about 800) dwarfs the millimetre columns (standard deviations of about 2 to 14), so k-means clusters almost only on weight: ARI 0.331. Adelie and Chinstrap weigh about the same, so weight cannot separate them. After standardising, every measurement counts equally, and bill length (long in Chinstrap, short in Adelie) can do its job: ARI 0.793. Gentoo is found perfectly (123 of 123 in one cluster); the remaining confusion is 24 Adelie placed with the Chinstrap and 5 Chinstrap placed with the Adelie, two species that overlap in body shape. Real segmentation has no labels to check against, so you would judge clusters by stability and usefulness instead.

**Complexity:** `O(n k d)` per iteration, 10 restarts: milliseconds for 342 rows.

**Common mistakes:** Using accuracy to compare clusters with labels (cluster numbers are arbitrary; use ARI or a crosstab). Forgetting `n_init` (one unlucky start can give a poor local optimum). Scaling with statistics from data you will later evaluate on, in a supervised setting.

**Learn more:** [ai-unsupervised](https://mahsa-agz.github.io/ds-handbook/#ai-unsupervised), [ai-features-imbalance](https://mahsa-agz.github.io/ds-handbook/#ai-features-imbalance)

</details>

---
### Q3. Thirty columns, how many directions?

*easy, about 6 min*

Use the breast cancer data (30 features).

1. Run PCA on the **raw** features. Print the share of variance explained by the first component and the feature with the largest absolute weight in it.
2. Run PCA on **standardised** features. Print the share explained by PC1, by PC1 + PC2, and the number of components needed to keep 95% of the variance.

What does the raw result really tell you? When would you use PCA before a model, and what do you lose?

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA

# your code here

<details><summary><b>Hint 1</b></summary>

Signal: many correlated numeric columns. Pattern: PCA (directions of maximum variance), which is scale-sensitive.

</details>

<details><summary><b>Hint 2</b></summary>

1. `PCA().fit(X)`; `explained_variance_ratio_`; `components_[0]` holds PC1's weights. 2. Scale first. 3. `np.cumsum` and the first index where it reaches 0.95.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data

pca_raw = PCA().fit(X)
top = data.feature_names[np.argmax(np.abs(pca_raw.components_[0]))]
print("unscaled: PC1 explains", round(pca_raw.explained_variance_ratio_[0], 3), "| biggest weight:", top)
# out: unscaled: PC1 explains 0.982 | biggest weight: worst area

pca = PCA().fit(StandardScaler().fit_transform(X))
cum = np.cumsum(pca.explained_variance_ratio_)
print("scaled: PC1", round(pca.explained_variance_ratio_[0], 3), " PC1+PC2", round(cum[1], 3))
print("components for 95% of variance:", int(np.searchsorted(cum, 0.95) + 1), "of", X.shape[1])
# out: scaled: PC1 0.443  PC1+PC2 0.632
# out: components for 95% of variance: 10 of 30
```

**Why:** On raw data PC1 explains 98.2% of the variance, but that only says 'worst area' has by far the largest numbers (areas are in the hundreds to thousands, smoothness is around 0.1). PCA maximises variance, so unscaled it just finds the biggest unit. On standardised data PC1 explains 44.3% and the first two 63.2%, and 10 of the 30 components keep 95% of the information, because the features are strongly correlated (radius, perimeter and area measure nearly the same thing). Use PCA before a model to remove collinearity, to speed up distance-based models, or to plot data in 2-D. You lose interpretability (each component mixes all features) and possibly signal: PCA ignores the target, so a low-variance direction can still be the predictive one.

**Complexity:** PCA via SVD is `O(n d^2)` for `n > d`: instant for 569 x 30.

**Common mistakes:** Running PCA on unscaled data with mixed units. Fitting PCA on all data before a train/test split (fit it inside the pipeline). Explaining components as if they were original features.

**Learn more:** [ai-unsupervised](https://mahsa-agz.github.io/ds-handbook/#ai-unsupervised)

</details>

---
### Q4. One step of the algorithm

*easy, about 6 min*

Write `kmeans_step(X, centers)` with numpy: assign each row of X to its nearest center (squared Euclidean distance; on a tie, the lower index wins), then return the new centers, the mean of the points assigned to each. A center that gets no points stays where it is.

Example: points `(0,0), (0,2), (10,0), (10,2)` and centers `(1,1), (9,1)` give `(0,1), (10,1)`.

Look at the second test: what does it say about k-means and initialisation?

In [ ]:
def kmeans_step(X, centers):
    # your code here
    pass

# tests
pts = [[0, 0], [0, 2], [10, 0], [10, 2]]
check(kmeans_step, [
    ((pts, [[1, 1], [9, 1]]), [[0, 1], [10, 1]]),
    ((pts, [[0, 0], [0, 2]]), [[5, 0], [5, 2]]),            # a poor start: splits top/bottom instead of left/right
    ((pts, [[0, 1], [100, 100]]), [[5, 1], [100, 100]]),    # second center gets no points: it stays put
    (([[1], [2], [3], [10], [11]], [[0], [20]]), [[4], [11]]),   # 10 is equally far from 0 and 20: tie -> center 0
])

<details><summary><b>Hint 1</b></summary>

Signal: 'assign, then average'. Pattern: Lloyd's algorithm for k-means.

</details>

<details><summary><b>Hint 2</b></summary>

1. Distances with broadcasting: `X[:, None, :] - centers[None, :, :]`, square, sum over the last axis. 2. `argmin(axis=1)` gives labels (ties go to the first). 3. For each j, mean of `X[labels == j]` if any.

</details>

<details><summary><b>Solution</b></summary>

```python
def kmeans_step(X, centers):
    """One Lloyd iteration: assign each point to the nearest center, then move each center to its points' mean."""
    X, centers = np.asarray(X, dtype=float), np.asarray(centers, dtype=float)
    dist = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)   # (n, k) squared distances
    labels = dist.argmin(axis=1)
    new = centers.copy()                                               # an empty cluster keeps its center
    for j in range(len(centers)):
        if np.any(labels == j):
            new[j] = X[labels == j].mean(axis=0)
    return new
```

**Why:** Both steps can only lower the inertia, which is why k-means converges. The second test starts with centers at `(0,0)` and `(0,2)`: the step splits the data into a bottom pair and a top pair, giving centers `(5,0)` and `(5,2)`. That is already a fixed point (another step changes nothing), with inertia 100, while the natural left/right split has inertia 4. So k-means can get stuck in a bad local optimum; k-means++ initialisation and several restarts (`n_init`) are the cure.

**Complexity:** `O(n k d)` time and `O(n k)` memory for the distance matrix per step.

**Common mistakes:** Taking the square root (not needed for argmin). Crashing on an empty cluster (mean of nothing is NaN). Looping over points in Python instead of broadcasting.

**Learn more:** [ai-unsupervised](https://mahsa-agz.github.io/ds-handbook/#ai-unsupervised), [cheat-numpy](https://mahsa-agz.github.io/ds-handbook/#cheat-numpy)

</details>

---
### Q5. Prepare customers for clustering (review)

*easy, about 3 min*

Marketing wants customer segments. Columns: `age`, `annual_income` (heavily right-skewed), `orders_last_year` (many zeros, a few heavy buyers), `country` (40 values), `signup_date`. What feature work do you do before k-means, and why?

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: distance-based method plus mixed, skewed, categorical features. Pattern: feature engineering for distances (transform, scale, encode).

</details>

<details><summary><b>Hint 2</b></summary>

Log-transform skewed money and count columns, turn dates into durations, handle the 40 countries (group or use regions), then standardise everything.

</details>

<details><summary><b>Solution</b></summary>

**Model answer:** "k-means uses Euclidean distance, so every feature must be numeric, on a comparable scale, and not dominated by a few extreme values.

- `annual_income`: `log1p` (otherwise a few rich customers decide the clusters), then standardise.
- `orders_last_year`: `log1p` or clip at a high percentile; maybe add a 0/1 'has ordered' flag.
- `signup_date`: convert to tenure in months; a raw date is not a meaningful distance.
- `country`: 40 one-hot columns would dominate the distance and make the clusters about geography. Group into regions or markets, or cluster within each market. If categorical data matters, use k-prototypes or Gower distance instead.
- Standardise all numeric features; consider PCA if many are correlated.
- Choose features by the business question (behaviour versus demographics), then check that the segments are stable across random seeds and samples and that they are explainable."

**Why:** It reuses the day 9 lessons (skew, encoding, missing meaning) for a distance-based method, where scale decides everything.

**Common mistakes:** One-hot encoding 40 countries next to 3 numeric columns. Clustering raw income. Treating the signup date as a number.

**Learn more:** [ai-features-imbalance](https://mahsa-agz.github.io/ds-handbook/#ai-features-imbalance), [ai-unsupervised](https://mahsa-agz.github.io/ds-handbook/#ai-unsupervised)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Done for today.** Finish with the [theory flashcards for day 10](https://mahsa-agz.github.io/ds-handbook/#day-10).